# DRM · Lab 04 — Assignment 03 Sheet (GRADED)
## Option Pricing and Synthetic Portfolios on **your allotted Nifty 50 stock**

**Course:** DRM (<ECON F354>) · <Semester 1 - 2026-27>
**Instructor:** <Prof. Aditya Sharma> · BITS Pilani
**Total marks:** 45 **Deadline:** `<DD Mon 2026, Midnight>`

---

### What this assignment is

You have to choose one company from the **Nifty 50**. On that stock you will

- estimate volatility from two years of daily prices (**data collection**),
- build a **Black-Scholes-Merton** pricer and a **binomial tree** pricer for European options and
  show that the second converges to the first,
- engineer a **synthetic long call** from the stock and a put using put-call parity, and compare
  it with the actual call — today, over the last three months, and trade-by-trade over two years.

### Before you start

1. Fill in the identification cell below, **including the ticker selected by you**.
   **A submission without it will not be graded.**
2. Write your code in the cells marked `# ---- YOUR CODE HERE ----` and your written answers in
   the markdown cells marked `**Your answer:**`. Do not delete the question cells.
3. **Marks are for interpretation as much as for code.** A correct plot with no commentary earns
   roughly half the marks for that part. Every question that asks "why" or "comment" is looking
   for economics, not for a restatement of what the chart shows.
4. The rubric for each question is printed **directly under that question**. Read it before you
   answer — it tells you exactly what earns the marks.

### Submission

- `Kernel → Restart & Run All`, confirm there are **no errors and all outputs are visible**.
- Save as **`DRM_Lab04_<CampusID>.ipynb`** — e.g. `DRM_Lab04_2023A7PS0123P.ipynb`.
- Upload to the Lab 04 assignment on **Nalanda**. Do not email it; do not open a pull request.

### Marks

| Q | Topic | Marks |
|---|---|---|
| Q1 | Data collection, log returns and annualised volatility | 2 |
| Q2 | Black-Scholes-Merton pricer and Greeks | 5 |
| Q3 | Binomial tree pricer and convergence to BSM | 5 |
| Q4 | Synthetic long call from put-call parity | 5 |
| Q5 | Historical arbitrage simulation: synthetic vs actual call | 8 |
| Q6 | Short answers | 5 |
| | **Total** | **30** |

---
## Evaluation rubric

Each question below carries its own **criterion table** showing how its marks are split. Every
criterion in those tables is graded on the same four-level scale:

| Level | Share of the criterion's marks | Code and output | Written interpretation |
|---|---|---|---|
| **Complete** | 100% | Runs on `Restart & Run All`; result is correct; the required check (assertion, benchmark, table) is present and passes; figure has a title, axis labels with units and a legend | Explains the **mechanism** in economic terms, refers to the student's **own numbers**, and states data provenance (live or simulated) |
| **Substantial** | about 70% | Correct method with one minor slip (a unit, a day-count, an unlabelled axis, a missing check) | Right conclusion, but the mechanism is thin or the numbers are not quoted |
| **Partial** | about 40% | Runs, but a formula or convention is wrong, or part of the required output is missing | Restates what the chart shows without explaining why; or generic text that could belong to any stock |
| **None** | 0% | Missing, does not run, or hard-coded output | Blank, or copied question text |

**How the levels turn into marks.** A criterion worth 2 marks graded *Substantial* earns about
1.5; graded *Partial*, about 1. Marks are awarded in steps of 0.5.

**Notebook-wide rules** (applied after the question marks are added up):

| Notebook-wide issue | Effect |
|---|---|
| Identification cell or honour statement not completed | Not graded |
| `Restart & Run All` raises an error | Cells after the first error are graded at most at the **Partial** level |
| Outputs not visible in the saved file | The affected part is graded as **None** |
| Simulated (`[SIM ]`) or model-generated numbers presented as market data | −2 marks |
| A required source is missing (each of the look-ups listed in the checklist) | −1 mark each |
| Analysis run on a stock other than the one allotted | Not graded |
| AI or outside help used but not disclosed in the declaration | Referred as an integrity violation |
| Late submission | -5 Marks |

**What "interpretation" means here.** Three tests are applied to every written answer:
(i) does it name a mechanism (financing cost, convexity, discreteness of the tree, overlap of
trades…) rather than describe a picture; (ii) does it quote the number *your* notebook produced;
(iii) is it honest about what is model-generated and what is observed.

In [ ]:
# ==================================================================================
#  IDENTIFICATION — fill this in. Submissions without it will not be graded.
# ==================================================================================
NAME       = ""          # e.g. "Ananya Sharma"
CAMPUS_ID  = ""          # e.g. "2023A7PS0123P"
EMAIL      = ""

# The Nifty 50 company allotted to you. Everything below is computed on THIS stock.
TICKER     = ""          # e.g. "HDFCBANK.NS"  (Yahoo symbol = NSE symbol + ".NS")

# Honour statement. Type "I agree" to confirm that the work below is your own and that any
# outside help, source or AI assistance is disclosed in the final cell of this notebook.
HONOUR     = ""

assert NAME and CAMPUS_ID and TICKER and HONOUR.strip().lower() == "i agree", \
    "Fill in NAME, CAMPUS_ID, TICKER and type 'I agree' in HONOUR before running the rest."
print(f"{NAME} | {CAMPUS_ID} | {TICKER}")

---
## Provided setup — run these two cells as-is

The data layer is given to you so that a network failure never blocks the assignment. It tries the
live API first and falls back to a seeded simulator, exactly as in the worked example. **If your
loaders report `[SIM ]`, say so in your write-up** and interpret your results accordingly — you
will not be penalised for working offline, but you will be penalised for presenting simulated
output as though it were market data.

Two conventions are fixed here so that everybody's answers are comparable:

- **Window.** `stock.loc[WIN_START:]` is the two-year analysis window. About four extra months are
  downloaded before it so that rolling volatility is already defined on the first day of the window.
- **Prices.** Closing prices are **not** adjusted for dividends (`auto_adjust=False`), because an
  option's strike is not adjusted for ordinary dividends either.

In [ ]:
# !pip install -q yfinance pandas_datareader        # uncomment in Colab if needed

import warnings, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

warnings.filterwarnings("ignore")
plt.rcParams.update({
    "figure.figsize": (11, 4.4), "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold", "legend.frameon": False,
})
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

SEED      = 2026
TODAY     = dt.date.today()
END       = TODAY.isoformat()
WIN_START = (TODAY - dt.timedelta(days=730)).isoformat()          # start of the 2-year analysis window
START     = (TODAY - dt.timedelta(days=730 + 120)).isoformat()    # + ~4 months of warm-up for rolling volatility
DATA_MODE = "auto"          # "auto" | "live" | "offline"
IS_SIM = {}

def ttl(title, key=None):
    return title + ("   —   [SIMULATED DATA]" if IS_SIM.get(key) else "")

print(f"pandas {pd.__version__} | download {START} -> {END} | analysis window from {WIN_START} | DATA_MODE = {DATA_MODE}")

In [ ]:
def fetch_yahoo(ticker, start=START, end=END):
    import yfinance as yf
    df = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    if df is None or len(df) == 0:
        raise RuntimeError("empty frame")
    if isinstance(df.columns, pd.MultiIndex):
        df = df.droplevel(1, axis=1)
    s = df["Close"].astype(float).dropna()
    s.index = pd.to_datetime(s.index).tz_localize(None)
    return s.rename(ticker)

def fetch_fred(series, start=START, end=END):
    from pandas_datareader import data as pdr
    s = pdr.DataReader(series, "fred", start, end).iloc[:, 0].astype(float).dropna()
    s.index = pd.to_datetime(s.index)
    return s.rename(series)

def sim_gbm_series(name, S0, mu, sigma, seed, start=START, end=END):
    idx = pd.bdate_range(start, end)
    rng = np.random.default_rng(seed)
    dtt = 1 / 252
    incr = (mu - 0.5 * sigma**2) * dtt + sigma * np.sqrt(dtt) * rng.standard_normal(len(idx))
    return pd.Series(S0 * np.exp(np.cumsum(incr)), index=idx, name=name)

def sim_ou_series(name, x0, level, kappa, vol, seed, start=START, end=END):
    idx = pd.bdate_range(start, end)
    rng = np.random.default_rng(seed)
    dtt, x, out = 1 / 252, x0, np.empty(len(idx))
    for i in range(len(idx)):
        x += kappa * (level - x) * dtt + vol * np.sqrt(dtt) * rng.standard_normal()
        out[i] = x
    return pd.Series(out, index=idx, name=name)

SIM_ANCHOR = "2021-01-01"      # simulated paths start here, so they do not change from one day to the next

def sim_window(s, S0=None):
    """Cut a simulated path to the download window; optionally rescale it to start at S0."""
    s = s.loc[START:]
    return s if S0 is None else (s / s.iloc[0] * S0).rename(s.name)

def load(key, live_fn, sim_fn):
    if DATA_MODE != "offline":
        try:
            s = live_fn(); IS_SIM[key] = False
            print(f"  [LIVE] {key:<10s} {len(s):>5d} obs  {s.index.min().date()} -> {s.index.max().date()}")
            return s
        except Exception as e:
            if DATA_MODE == "live":
                raise
            print(f"  [SIM ] {key:<10s} live fetch failed ({type(e).__name__}) -> simulated")
    else:
        print(f"  [SIM ] {key:<10s} DATA_MODE=offline")
    IS_SIM[key] = True
    return sim_fn()

print("Loading\n" + "-" * 70)
stock = load("stock", lambda: fetch_yahoo(TICKER),
                      lambda: sim_window(sim_gbm_series(TICKER, 1.0, 0.12, 0.240, SEED + 137, start=SIM_ANCHOR), 1_450.0))
inr_r = load("inr_r", lambda: fetch_fred("INDIRLTLT01STM") / 100.0,
                      lambda: sim_window(sim_ou_series("INR rate", 0.066, 0.066, 1.0, 0.007, SEED + 74, start=SIM_ANCHOR)))
print("-" * 70)

inr_r = inr_r.reindex(stock.index).ffill().bfill()
print(f"{TICKER} {stock.iloc[-1]:,.2f} | r_INR {inr_r.iloc[-1]:.3%} | "
      f"{len(stock.loc[WIN_START:])} trading days in the 2-year window")

---
## Q1 · Data collection, log returns and annualised volatility · **[2 marks]**

**(a)** Compute daily **log returns** for your stock. Produce a 2×2 figure over the two-year
window showing: the closing price; the daily log returns; **rolling annualised volatility** on a
21-day and a 63-day window with the full-sample value drawn as a horizontal line; and a histogram
of daily log returns with a normal density of the same mean and standard deviation overlaid.
Label axes with units.

**(b)** Build a summary table containing: number of observations, start and end dates, total
return, **annualised volatility** $\hat\sigma = \text{sd}(\text{log returns})\times\sqrt{252}$,
annualised mean log return, skewness, excess kurtosis and **maximum drawdown**. Store the
volatility in a variable called `SIGMA_HAT` — every later question uses it.

**(c)** In the markdown cell below: report $\hat\sigma$, and the lowest and highest values that the
21-day rolling volatility took in the window. Then answer: **BSM assumes a single constant
$\sigma$. Given your rolling-volatility plot, which estimate would you feed into the pricer for a
one-month option priced today — the two-year figure, the latest 21-day figure, or something else —
and what is the cost of getting it wrong?**

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Log (not simple) returns; all four panels present, correctly annualised, labelled with units; `[SIMULATED DATA]` flag carried in titles if applicable |  |
| (b) | Every statistic present and correct; volatility uses the sample standard deviation and $\sqrt{252}$; drawdown is peak-to-trough and negative; `SIGMA_HAT` defined | 1 |
| (c) | Quotes own $\hat\sigma$ and the rolling range; recognises that volatility is time-varying and clustered; makes a reasoned choice of estimate for a one-month option and links a volatility error to a pricing error through vega | 1 |

In [ ]:
# ---- YOUR CODE HERE ----  Q1 (a) and (b)

def max_drawdown(series):
    """Return the maximum peak-to-trough decline as a NEGATIVE fraction, e.g. -0.32."""
    # TODO
    raise NotImplementedError

def summary_stats(series):
    """Return a dict with n_obs, start, end, total_return, ann_vol, ann_drift, skew, excess_kurt, max_dd."""
    # TODO
    raise NotImplementedError

SIGMA_HAT = None     # TODO: annualised volatility over the 2-year window (a float, e.g. 0.2350)

# TODO: build the 2x2 figure, then assemble and display the summary table.

**Your answer (Q1c):**

*Data provenance (live or simulated):*

*$\hat\sigma$ (2-year) = ______ ; 21-day rolling range = ______ to ______*

*Which estimate you would use for a one-month option, and why:*

*Cost of getting $\sigma$ wrong:*

---
## Q2 · Black-Scholes-Merton pricer and Greeks · **[5 marks]**

For a European option on a stock paying a continuous dividend yield $q$:

$$C = S e^{-qT}N(d_1) - K e^{-rT}N(d_2), \qquad P = K e^{-rT}N(-d_2) - S e^{-qT}N(-d_1)$$

$$d_1 = \frac{\ln(S/K) + (r - q + \tfrac12\sigma^2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

**(a)** Implement `bsm_price(S, K, T, r, sigma, q=0, kind="call")` and
`bsm_greeks(S, K, T, r, sigma, q=0, kind="call")` returning delta, gamma, vega and theta. The
function must accept NumPy arrays for `S` and `sigma`. **Include three assertions — numerical
checks, not comments:**
1. the textbook benchmark $S=42,\ K=40,\ r=10\%,\ \sigma=20\%,\ T=0.5$ gives $C=4.76$, $P=0.81$;
2. **put-call parity** $C - P = Se^{-qT} - Ke^{-rT}$ holds to $10^{-8}$ for an arbitrary input with $q \neq 0$;
3. your analytic delta and vega agree with a central **finite difference** of your own price function.

**(b)** Set the **user inputs** for your stock: $S$ = latest close, $K = S$ (at-the-money),
$T = 30/365$, $r$ = latest value of `inr_r`, $\sigma$ = `SIGMA_HAT`, and a dividend yield $q$ that
you **look up and cite**. Report a table with the call premium, the put premium and the four
Greeks of each, with units stated (vega per one volatility point, theta per calendar day).

**(c)** Plot **Delta against stock price** (from 70% to 130% of the strike) for the call and the
put, for three maturities: 1 week, 1 month, 3 months. Plot **Vega against volatility** (from 5% to
80%) for three strikes: 90%, 100% and 110% of spot, at $T = 30/365$.

**(d)** In the markdown cell below, answer all three:
- Why does the delta curve steepen around the strike as expiry approaches, and what does that
  imply for someone hedging a short ATM option in its final week?
- Your ATM call delta is slightly **above** 0.5 even though the option is exactly at the money.
  Show from the formula why.
- Describe the vega-vs-volatility plot: why is the ATM line almost flat while the OTM and ITM lines
  rise from zero? Why do the call and the put with the same strike and expiry have the same vega?

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Correct call and put with dividend yield; vectorised; all three assertions present **and passing** (benchmark, parity, finite difference) | 1 |
| (b) | Inputs taken from own data, not typed in; $q$ cited with a named source; premiums and Greeks tabulated with units | 1 |
| (c) | Delta-vs-$S$ plot (call and put, three maturities) | 1 |
| (c) | Vega-vs-$\sigma$ plot (three strikes), axes in sensible units | 1 |
| (d) | Gamma concentration near expiry and its hedging consequence; $d_1 = (r - q + \sigma^2/2)\sqrt{T}/\sigma > 0$ at the money; vega shape explained and call-put vega equality derived from parity | 1 |

In [ ]:
# ---- YOUR CODE HERE ----  Q2 (a)

def bsm_price(S, K, T, r, sigma, q=0.0, kind="call"):
    # TODO
    raise NotImplementedError

def bsm_greeks(S, K, T, r, sigma, q=0.0, kind="call"):
    """Return a dict: delta, gamma, vega (per 1.00 of volatility), theta (per year)."""
    # TODO
    raise NotImplementedError

# TODO: assertion 1 - textbook benchmark (C = 4.76, P = 0.81)
# TODO: assertion 2 - put-call parity to 1e-8 with q != 0
# TODO: assertion 3 - analytic delta and vega vs central finite differences

In [ ]:
# ---- YOUR CODE HERE ----  Q2 (b) and (c)
Q_DIV        = None     # TODO: dividend yield of YOUR stock as a decimal, e.g. 0.011
Q_DIV_SOURCE = ""       # TODO: where you got it and the date you checked

# TODO: user inputs S0, K, T, r, sigma taken from the data above (do not type the numbers in)
# TODO: table of premiums and Greeks for the ATM call and put
# TODO: figure - Delta vs S (call and put; 1 week, 1 month, 3 months)
# TODO: figure - Vega vs sigma (K = 90%, 100%, 110% of spot; T = 30/365)

**Your answer (Q2):**

*Dividend yield used and source:*

*(d-i) Why delta steepens near expiry, and what it means for a hedger:*

*(d-ii) Why the ATM call delta exceeds 0.5:*

*(d-iii) Shape of vega vs volatility, and why call vega = put vega:*

---
## Q3 · Binomial tree pricer and convergence to BSM · **[5 marks]**

Use the Cox-Ross-Rubinstein parameterisation with $N$ steps of length $\Delta t = T/N$:

$$u = e^{\sigma\sqrt{\Delta t}}, \qquad d = 1/u, \qquad p = \frac{e^{(r-q)\Delta t} - d}{u - d}$$

**(a)** Implement `binomial_price(S, K, T, r, sigma, N, q=0, kind="call")` for **European**
options by backward induction. The function must raise an error if the risk-neutral probability
is not strictly between 0 and 1. Do not use the closed-form binomial sum, and do not loop over
nodes one at a time in Python — work on a whole time-slice as an array.

**(b)** With the same user inputs as Q2(b), price the ATM call and put with **$N = 50$** and
$N = 500$. Report a table of binomial price, BSM price, and the difference in rupees and in
basis points of the BSM price. **Assert** that your tree satisfies put-call parity to $10^{-8}$
at $N=50$ — it must, for any $N$.

**(c)** Convergence. Compute the call price for every $N$ from 1 to 400 and plot (i) the binomial
price against $N$ with the BSM value as a horizontal line, and (ii) the absolute error against $N$
on log-log axes with a reference line of slope $-1$. Repeat (i) for a strike **5% out of the
money**. Add a table for $N \in \{10, 25, 50, 51, 100, 101, 200, 201, 400\}$.

**(d)** In the markdown cell below, answer all three:
- At what rate does the error fall with $N$? Read it off your log-log plot, and say how many steps
  you need for the price to be within ₹0.01 of BSM.
- The ATM error **zig-zags** between odd and even $N$, and the OTM error moves in **waves**. What
  is the cause? Name one simple fix and show numerically that it works.
- Parity holds exactly in your tree even at $N=5$, where the price itself is poor. Why?

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Correct CRR backward induction, vectorised across a time-slice, dividend yield in $p$, validity check on $p$; $N \ge 50$ supported | 1 |
| (b) | Table at $N=50$ and $N=500$ against BSM; parity assertion on the tree present and passing | 1 |
| (c) | Both convergence plots, the OTM comparison and the table; log-log axes and reference slope | 1 |
| (d) | $O(1/N)$ rate read from own plot and translated into a step count; oscillation attributed to the position of the strike relative to the terminal nodes, with a fix demonstrated; exact parity explained by linearity of risk-neutral expectation | 2 |

In [ ]:
# ---- YOUR CODE HERE ----  Q3 (a) and (b)

def binomial_price(S, K, T, r, sigma, N=50, q=0.0, kind="call"):
    # TODO: CRR tree, European exercise, backward induction on arrays
    raise NotImplementedError

# TODO: table - binomial (N = 50, 500) vs BSM for the ATM call and put
# TODO: assert put-call parity on the tree at N = 50

In [ ]:
# ---- YOUR CODE HERE ----  Q3 (c): convergence plots and table

**Your answer (Q3d):**

*Rate of convergence and the number of steps needed for ₹0.01 accuracy:*

*Cause of the zig-zag (ATM) and the waves (OTM); one fix and the evidence that it works:*

*Why parity is exact on the tree for any $N$:*

---
## Q4 · Synthetic long call from put-call parity · **[5 marks]**

Put-call parity for European options with the same strike and expiry:

$$C - P = S e^{-qT} - K e^{-rT} \quad\Longleftrightarrow\quad C = S e^{-qT} + P - K e^{-rT}$$

The strategy to build is **long stock + long put**, with the put struck at the same level and
expiring on the same day as the call you are replicating.

**(a) Cost and payoff today.** Take the ATM call from Q2(b) ($K = S$, $T = 30/365$). Using your
own pricer, tabulate the **cost today** and the **payoff at expiry** of three portfolios:
(1) the actual call; (2) the synthetic *stock + put*; (3) the *financed* synthetic
*stock + put − loan of $Ke^{-rT}$*. Plot the expiry **payoff** and the expiry **profit** of all
three against $S_T$ from 80% to 120% of the strike.

**(b) An observed quote.** Obtain the **observed** premiums of a near-the-money call and put on
your stock with the same strike and expiry — from the NSE option chain, your broker's terminal or
an API — together with the spot price **at the same moment**. Record strike, expiry date,
timestamp and source. Compute:
- the **parity gap** $C^{obs} - P^{obs} - (S e^{-q\tau} - K e^{-r\tau})$ in rupees and in bps of spot;
- the **implied financing rate** $r_{\text{implied}} = -\frac{1}{\tau}\ln\!\big[(S e^{-q\tau} - C^{obs} + P^{obs})/K\big]$;
- the **implied volatility** of the call and of the put, by root-finding on your `bsm_price`.

Compare the implied volatilities with `SIGMA_HAT`, and the implied financing rate with `inr_r`.

**(c) Track over the last three months.** Take the last three consecutive 30-day cycles ending
on the latest date in your data. In each cycle, open the actual call and the synthetic at the
close of the first day ($K$ = that day's close), **mark both to model every day** to expiry
(using that day's spot, rate, remaining time and 21-day rolling volatility), then roll. Plot the
cumulative P&L per share of the actual call, the synthetic and the financed synthetic. Report a
per-cycle table: entry and expiry dates, $S_0$, $S_T$, $C_0$, $P_0$, and the P&L and return on
capital of each portfolio.

**(d)** In the markdown cell below, **explain the discrepancies**:
- The synthetic *stock + put* costs far more than the call and pays $K$ more at expiry. Which
  portfolio is the true replica of the call, and what does the difference in **profit** between
  portfolios (1) and (2) represent? Give it as a formula and as a rupee figure from your table.
- In (c) the gap between the synthetic and the call is not constant through a cycle. Derive what
  it equals on any day, and confirm it against your series.
- In (b), is your parity gap large enough to trade? Name the arbitrage you would put on
  (conversion or reversal), and give **three** reasons the gap may not be capturable.
- Your call and put implied volatilities differ from `SIGMA_HAT`. Which way, and why is that the
  normal state of affairs?

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Cost and payoff table for all three portfolios; payoff and profit diagrams; shows payoff(2) = payoff(1) + $K$ and cost(3) = cost(1) | 1 |
| (b) | Real quote with strike, expiry, timestamp and source, spot taken at the same moment; parity gap, implied financing rate and both implied volatilities computed by root-finding. *If no live quote could be obtained, a clearly labelled model-generated quote earns at most 1* | 1 |
| (c) | Three non-overlapping cycles; daily mark-to-model with shrinking time to expiry and no look-ahead; cumulative P&L chart and per-cycle table with return on capital | 1 |
| (d) | Identifies the financed portfolio as the replica and the profit gap as the financing cost $S_0 - Ke^{-rT}$ (less dividends); derives the daily gap from parity; realistic view on capturing the observed gap (bid-ask, margin and short-selling constraints, asynchronous quotes, dividends, taxes and fees); implied-vs-realised volatility explained | 2 |

In [ ]:
# ---- YOUR CODE HERE ----  Q4 (a): cost table, payoff and profit diagrams

In [ ]:
# ---- YOUR CODE HERE ----  Q4 (b): observed quote, parity gap, implied financing rate, implied vols
C_OBS        = None     # TODO: observed call premium (INR)
P_OBS        = None     # TODO: observed put premium, same strike and expiry (INR)
K_OBS        = None     # TODO: the listed strike you used (INR)
S_OBS        = None     # TODO: spot price at the SAME moment as the option quotes (INR)
EXPIRY_OBS   = ""       # TODO: expiry date of the contract, "YYYY-MM-DD"
QUOTE_ASOF   = ""       # TODO: timestamp of the quotes, e.g. "2026-11-20 15:25 IST"
QUOTE_SOURCE = ""       # TODO: where you got them

def implied_vol(price, S, K, T, r, q=0.0, kind="call"):
    # TODO: root-find sigma such that bsm_price(...) == price
    raise NotImplementedError

# TODO: parity gap (INR and bps of spot), implied financing rate, implied vol of call and put

In [ ]:
# ---- YOUR CODE HERE ----  Q4 (c): track the actual call and the synthetic over the last 3 months

**Your answer (Q4d):**

*Data provenance — which numbers above are observed and which are model-generated:*

*Which portfolio replicates the call; what the profit gap between (1) and (2) is (formula and ₹):*

*What the synthetic-minus-call gap equals on any day within a cycle:*

*Is the observed parity gap tradable? Which arbitrage, and three obstacles:*

*Implied vs realised volatility:*

---
## Q5 · Historical arbitrage simulation: synthetic vs actual call · **[8 marks]**

Repeat the comparison of Q4 on **every trading day** of the two-year window.

**Fixed parameters.** Time to expiry: 30 calendar days. Strike: at the money ($K$ = that day's
close). Horizon: two years of daily data. Each day you open two positions and hold both to expiry:

- **Strategy A** — buy the actual call.
- **Strategy B** — synthetic call: buy the stock and buy the put.

**(a) Build the trade panel** with one row per entry date and columns
`S0, K, r, T, expiry, S_T, sigma_call, sigma_put, C0, P0, pnl_call, pnl_syn, pnl_fin, cap_call, cap_syn`,
where `pnl_fin` is the P&L of the financed synthetic of Q4 and `cap_*` is the capital paid on day 0.
Rules:
- the expiry is the last trading day on or before entry + 30 calendar days; drop entry dates whose
  expiry lies beyond your data, and report how many trades remain (it will be a little under 500);
- **no look-ahead** — the volatility used on a date may only use returns up to that date;
- the closes are not dividend-adjusted, so credit the stock leg of strategy B with dividend income
  of $S_0(e^{qT}-1)$ over the holding period, using your $q$ from Q2;
- if you have real historical option premiums (NSE bhavcopy, `jugaad-data`, `nsepython`), use them
  and say so. If not, generate $C_0$ and $P_0$ from your own pricer with $\sigma_t$ = the 21-day
  realised volatility, and add a **documented mean-reverting distortion** to the put's volatility to
  stand in for the parity gaps seen in real markets — **and state clearly in your write-up that the
  option premiums are synthetic.** Make every parameter of the distortion a named constant.

**(b) Performance comparison table**, per share, in exactly this layout:

| Metric | Actual Call | Synthetic Call | Winner |
|---|---|---|---|
| Total trades profitable | X / N | Y / N | |
| Avg. return per trade | ₹Z | ₹W | |
| Max single-trade profit | ₹P | ₹Q | |
| Max single-trade loss | ₹R | ₹S | |
| Avg. capital required | ₹ | ₹ | |
| Avg. return on capital (ROC) | % | % | |
| Std. dev. of ROC | % | % | |

Then report, for the **financed** synthetic against the call: the mean and standard deviation of
the P&L difference, the share of days on which the absolute difference exceeds a transaction-cost
band of **25 bps of spot**. Finally report a $t$-statistic for (i) the mean P&L of the actual call
and (ii) the mean financed-minus-call difference, **each computed two ways** — on all trades, and on
a **non-overlapping** subsample (every 21st trade).

**(c) Best and worst cases, and cost efficiency.** List the entry dates of the maximum profit and
maximum loss of each strategy, with the stock's move over the holding period, the volatility
priced in at entry, and the volatility **realised** over the holding period. Produce three
diagnostic plots:
1. P&L per trade of both strategies through time;
2. P&L against the stock's percentage move over the holding period (both strategies);
3. the distribution of **return on capital** for both strategies.

**(d)** Write **at least 150 words** answering all four of the following:
- Synthetic "wins" on rupee P&L. By how much per trade on average, what is that amount in
  economic terms, and what happens to it once the stock leg is financed? **Is anything left that
  you would call an arbitrage?**
- Which strategy wins on return on capital, and why is that the mirror image of the rupee result?
  Which would you rather hold if you had ₹1 lakh and a one-month bullish view?
- What do the best and worst trades have in common? Relate them to the stock's move and to
  realised-minus-priced volatility.
- You have nearly 500 trades. How many **independent** observations do you really have, and what
  does that do to any claim of statistical significance? List at least three costs or frictions
  your simulation still ignores.

| Part | Rubric — what earns the marks | Marks |
|---|---|---|
| (a) | Correct expiry mapping and trade count; volatility without look-ahead; P&L and capital definitions correct for all three portfolios; premium source documented (real or synthetic with named distortion parameters) | 2 |
| (b) | Table in the required layout with a justified "winner" in each row; financed comparison with cost band; $t$-statistics on all trades and on the non-overlapping subsample | 2 |
| (c) | Best/worst dates tabulated with move, priced and realised volatility; three diagnostic plots, labelled | 2 |
| (d) | ≥150 words; rupee advantage identified as financing cost ($\approx S_0(r-q)T$) and shown to vanish after financing, leaving only the distortion; leverage explains the ROC reversal; best/worst trades tied to large moves and to realised vs priced volatility; overlap reduces ~500 trades to ~24 independent ones; three or more ignored frictions | 2 |

In [ ]:
# ---- YOUR CODE HERE ----  Q5 (a)

def build_trades(spot, rate, q, horizon_days=30, vol_window=21, seed=SEED + 81):
    # TODO: returns a DataFrame indexed by entry date with the columns listed in Q5(a)
    raise NotImplementedError

In [ ]:
# ---- YOUR CODE HERE ----  Q5 (b): performance comparison table, financed comparison, t-statistics
TC_BPS = 25.0

In [ ]:
# ---- YOUR CODE HERE ----  Q5 (c): best / worst trades and the three diagnostic plots

**Your answer (Q5d) — at least 150 words:**

*Data provenance — underlying and option premiums:*

*The rupee advantage of the synthetic: size, meaning, and what financing does to it:*

*Return on capital, and which you would hold with ₹1 lakh and a bullish view:*

*What the best and worst trades have in common:*

*Independent observations, significance, and ignored frictions:*

---
## Q6 · Short answers · **[5 marks]**

Answer in **two to four sentences each**. Marks are for the mechanism, not the assertion.

1. Put-call parity is called **model-free**. What exactly does it assume, and what does it *not*
   assume that BSM does? If you observe a parity violation in single-stock options, does that tell
   you that BSM is wrong?

2. You priced every option in Q5 from **historical** volatility. In real markets, implied
   volatility is on average higher than the volatility subsequently realised. What does that do
   to the average P&L of a strategy that systematically **buys** options, and which of your Q5
   results would change in sign or size?

3. A classmate says: *"The binomial model is just a crude approximation to Black-Scholes, so
   there is no reason ever to use it."* Give two specific situations in which the tree is the
   better tool, and say what the tree can do in each that the formula cannot.

4. The synthetic call needs you to **own the stock**; the mirror-image synthetic **put**
   (long call + short stock) needs you to **short** it. Explain why, in practice, parity violations
   are larger and more persistent in one direction than the other, and which direction that is.

| Each sub-question | Marks |
|---|---|
| Correct conclusion **and** the mechanism behind it, within the length limit | 2 |
| Correct conclusion, mechanism vague or missing; or right mechanism with a wrong sign | 1 |
| Assertion only, or more than twice the length limit without a clear answer | 0 – 0.5 |

**Your answers (Q6):**

1.

2.

3.

4.

---
## Declaration and checklist

**Sources, collaborators and any AI assistance used** (be specific — which tool, for which part,
and what you did with the output). Full Disclosure of the prompt is expected and costs you nothing; undisclosed use
of AI will be considered as an integrity violation.

*Your declaration:*

---

### Before you submit, confirm every box

- [ ] `NAME`, `CAMPUS_ID`, `TICKER` and `HONOUR` are filled in and the identification cell runs
- [ ] The analysis is on the stock **allotted to you**
- [ ] `Kernel → Restart & Run All` completes with **no errors**
- [ ] Every plot and table is **visible in the saved file** (not an empty output cell)
- [ ] Every `**Your answer:**` cell is filled in — none left blank
- [ ] You stated, for each data series, whether it is **live or simulated**, and for each option
      price whether it is **observed or model-generated**
- [ ] All assertions in Q2 and Q3 are present and pass
- [ ] Sources cited for: dividend yield; the observed call and put quote (with timestamp)
- [ ] Q5(d) is at least 150 words
- [ ] Saved as `DRM_Lab04_<CampusID>.ipynb`
- [ ] Uploaded to the Lab 04 assignment on **Nalanda** before the deadline